# MRI Reconstruction | Portfolio edition

Coursework from **EEN180, Chalmers University of Technology**. All code was developed jointly in a pair, including Neo Barrdahl ([Nero-Vanetti](https://github.com/Nero-Vanetti)). The collaborator's name is not listed in this edition.

Start with the [project README](README.md) for the methods, selected results and setup. See the [review notes](../docs/REVIEW_NOTES.md) for implementation limitations and the small packaging changes made to this copy.

The original assignment structure, written answers and saved results are retained below. These are historical coursework outputs, not a claim that every scientific conclusion has been independently validated. Put `MRIdata.mat` beside this notebook before running it.


# EEN180: Image Reconstruction for Magnetic Resonance Imaging

You are given sets of data that correspond to the $k$-space of an MR image and in some tasks also the coil sensitivities, as well as some complementary data. The objective of this project is to reconstruct the MR images from the $k$-space data.

These instructions were originally written for MATLAB but have been converted to Python.

## Notes
- The project has a total of 10 pts. To pass, you need $\geq$ 5 pts.
- This is a group exercise. Discussions between groups are encouraged while copying of code among groups is strictly prohibited. Any evidence of plagiarism will be dealt with according to Chalmers regulations by the teaching team.
- The objective of this assignment is algorithm implementation. You are allowed to use some general purpose functions such as `fft` and `ifft`. Specific functions that are directly related to the assignment such as open-source code for SENSE reconstruction cannot be used. However, you should install the ```nibabel``` package (```pip install nibabel```).

## Theory

In MRI, the magnetic properties (spin) of the protons in our bodies are used to image tissues. A strong static magnetic field together with varying gradient fields cause radio waves to be emitted from the patient, varying in phase and frequency depending on the location in the body relative to the supplied gradient fields. These signals, i.e. frequencies and phases, can be measured with radiofrequency coils as data points in $k$-space. Using the Fourier transform, the $k$-space data can be reconstructed to image space.

MRI acquisition is typically slow, and parallel imaging techniques can be used to speed up acquisition. One common way is to subsample $k$-space by e.g. omitting every $2^{nd}$ $k$-space line. This causes an aliased image with "folding" or "wrap-around" artifacts.

**SENSE reconstruction** is a way to restore a proper image from subsampled data. Here, multiple coils are used and the data from each coil can be reconstructed into partial FOV images and combined ("unfolded") to a full FOV image. Each pixel in the folded image contains signal from two pixels at different locations $A$ and $B$ in the unfolded image $F$. Denote the two partial FOV folded images from coil 1 and 2 as $C_1$ and $C_2$, and the two coil sensitivity images $S_1$ and $S_2$. Then we can write:

$$C_1 = F(A) \cdot S_1(A) + F(B) \cdot S_1(B)$$
$$C_2 = F(A) \cdot S_2(A) + F(B) \cdot S_2(B)$$

where $F(A)$ and $F(B)$ is the unfolded image at the two locations, and $S_{\cdot}(A)$ and $S_{\cdot}(B)$ are the coil sensitivities at these locations. Using the above system of equations, we can solve for the unknown (and wanted) unfolded image $F$.

## Data

### About the $k$-space data
The file `MRIdata.mat` contain all data needed for the project. Assume that the data was acquired slice by slice, i.e. that no Fourier transform along the third dimension is needed to reconstruct the images. In tasks when data from multiple coils is available, the fourth dimension is used to indicate different coils.

### About the reconstructed images
The reconstructed images will typically be complex valued. In this project, you should only view and comment on the magnitude images. 

Hint: Use the function ```np.abs```.

### Import Required Libraries and Load Data

In [1]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fft2, ifft2, fftshift, ifftshift
import scipy.io as sio
import nibabel as nib

# Set Matplotlib to display images inline
%matplotlib inline
# Set default figure size for better visibility
plt.rcParams['figure.figsize'] = (12, 8)
# Use grayscale colormap by default for images
plt.rcParams['image.cmap'] = 'gray'

In [2]:
# Load the MRI data
try:
    data = sio.loadmat('MRIdata.mat')
    print("Successfully loaded MRI data")
    
    # Extract data
    kspace_full = data['kspace_full']
    kspace_coils = data['kspace_coils']
    kspace_subsampled = data['kspace_subsampled']
    coil_sensitivities = data['coil_sensitivities']
    nifti_info = data['nifti_info']
    
    print(f"Size k-space full: {kspace_full.shape}")
    print(f"Size k-space coils: {kspace_coils.shape}")
    print(f"Size k-space subsampled: {kspace_subsampled.shape}")
    print(f"Size coil sensitivities: {coil_sensitivities.shape}")
except Exception as e:
    print(f"Error loading data: {e}")

Successfully loaded MRI data
Size k-space full: (192, 129, 192)
Size k-space coils: (192, 129, 192, 2)
Size k-space subsampled: (96, 129, 192, 2)
Size coil sensitivities: (192, 129, 192, 2)


## Tasks

### Preparations

#### P1 (0 pts)
Install 3D Slicer from [https://download.slicer.org/](https://download.slicer.org/). You can find a lot of useful information on the 3D Slicer website, including tutorials, [https://www.slicer.org/wiki/Documentation/4.10/Training](https://www.slicer.org/wiki/Documentation/4.10/Training).

#### P2 (1 pts)
Make a function `field_of_view`, computing the field of view of the MR data. Inspect the image metadata in the variable `nifti_info` and apply your function to compute the field of view.

In [3]:
def field_of_view(image_sizes, pixel_sizes):
    """
    Compute the field of view based on the image sizes and pixel dimensions.
    
    Args:
    ----
    image_sizes : array_like
        The size in pixels in each axis (1D array [height, width, depth])
    pixel_sizes : array_like
        The size of pixels (mm) in each direction (1D array [height, width, depth])
    
    Returns:
    -------
    fov : ndarray
        The field-of-view (mm) in each direction (1D array [height, width, depth])
    """
    # YOUR CODE HERE
    image_sizes = np.asarray(image_sizes)
    pixel_sizes = np.asarray(pixel_sizes)

    fov = image_sizes * pixel_sizes
    return fov

In [4]:
# Extract metadata information
image_sizes = nifti_info['ImageSize'][0][0].squeeze()
pixel_sizes = nifti_info['PixelDimensions'][0][0].squeeze()

# Compute field of view
fov = field_of_view(image_sizes, pixel_sizes)
print(f"Image sizes: {image_sizes}")
print(f"Pixel sizes: {pixel_sizes}")
print(f"FOV: {fov[0]:.2f} x {fov[1]:.2f} x {fov[2]:.2f} mm")

Image sizes: [192 129 192]
Pixel sizes: [1.30208337 1.3999939  1.30208337]
FOV: 250.00 x 180.60 x 250.00 mm


### P3 (1 pts):

Write a function `save_nifti_image` that takes a complex-valued image as input and saves it as a magnitude image using the Nifti file format ([nifti.nimh.nih.gov/nifti-2](https://nifti.nimh.nih.gov/nifti-2)). You should use the `nibabel` package to save NIFTI files. Here is a guide on the package: [https://nipy.org/nibabel/gettingstarted.html](https://nipy.org/nibabel/gettingstarted.html)

Discuss why we need specific image file formats for medical images.


In [ ]:
def convert_metadata_to_dict(nifti_info):
    """
    Convert NIfTI metadata from structured array to dictionary.
    
    Args:
    ----
    nifti_info : structured array
        The NIfTI metadata as loaded from the .mat file.
    
    Returns:
    -------
    metadata_dict : dict
        The metadata as a dictionary.
    """
    metadata_dict = {}
    for key in nifti_info.dtype.names:
        metadata_dict[key] = nifti_info[key][0,0]
    return metadata_dict


def save_nifti_image(complex_im, filename, metadata):
    metadata_dict = convert_metadata_to_dict(metadata)

    magnitude_im = np.abs(complex_im)

    affine = np.eye(4)

    nifti_img = nib.Nifti1Image(magnitude_im, affine)

    if 'PixelDimensions' in metadata_dict:
        pix_dims = np.array(metadata_dict['PixelDimensions']).flatten()[:3]
        nifti_img.header.set_zooms(pix_dims)

    if not filename.endswith('.nii'):
        filename = filename + '.nii'

    nib.save(nifti_img, filename)
    print(f"Saved {filename}")

**Answer:** 
Medical images need specific file formats because they contain much more information than regular photos. In this lab, we worked with MRI volumes reconstructed from k-space, with multiple coils, slices and orientation information. Standard formats like PNG or JPEG can only store a single 2D image and cannot include metadata about voxel size, slice position, acquisition parameters or anatomy. Medical formats such as DICOM or NIfTI keep all of this information together so that software like 3D Slicer can display the data correctly in axial, sagittal and coronal views. They also avoid lossy compression, which is important because even small artifacts can affect analysis or diagnosis. Specialized formats ensure that the images can be interpreted consistently across systems and preserve the accuracy needed in medical imaging.


### Part A: Basic MR Reconstruction

The data to be used in this task is named `kspace_full`.

#### A1 (1 pts):

Write a function `k2im` for MR image reconstruction, given an array of $k$-space data. You are free to use `ifft2` and `fft2`. Remember to combine `ifft2` and `fft2` with `fftshift` and `ifftshift`, since the $k$-space data is computed for the frequencies:

$$k_{\text{row}} = -M, ..., M$$
$$k_{\text{col}} = -N, ..., N$$

while FFT2 computes the Fourier transform over frequencies $0, ..., 2M$ and $0, ..., 2N$.

In [6]:
def k2im(k):
    """
    Reconstruct an MR image from k-space (frequency domain).
    
    Args:
    ----
    k : ndarray
        The spatial frequencies in each direction (4D array, height x width x depth x n_coils)
    
    Returns:
    -------
    im : ndarray
        A complex-valued image with same shape as k
    """
    # YOUR CODE HERE
    im = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(k, axes=(0, 1)),axes=(0, 1)),axes=(0, 1))       
    return im

In [7]:
# Test with full k-space
k = np.squeeze(kspace_full)
im_full = k2im(k)

print(f"k-space shape: {k.shape}")
print(f"Reconstructed image shape: {im_full.shape}")


k-space shape: (192, 129, 192)
Reconstructed image shape: (192, 129, 192)


### A2 (0 pts): 
Reconstructed the image and use the function from task P3 and the supplied metadata in `nifti_info` to save it to a file.

In [8]:
# Select a single slice to visualize
slice_idx = 150 

# Create figure with subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot k-space
axes[0].imshow(np.abs(k[:, :, slice_idx]), cmap='gray')
axes[0].set_title(f'k-space (slice {slice_idx})')
axes[0].set_xlabel('Frequency (columns)')
axes[0].set_ylabel('Frequency (rows)')

# Plot reconstructed image
axes[1].imshow(np.abs(im_full[:, :, slice_idx]), cmap='gray')
axes[1].set_title(f'Reconstructed Image (slice {slice_idx})')
axes[1].set_xlabel('x (pixels)')
axes[1].set_ylabel('y (pixels)')

plt.tight_layout()
plt.show()

# YOUR CODE HERE
# Save the reconstructed image
save_nifti_image(im_full, 'reconstructed_full', nifti_info)


Saved reconstructed_full.nii


![Saved output from cell 16, figure 1](../assets/notebook-results/mri-cell-16-figure-0.png)


### A3 (1 pts): 3D Slicer Visualization
View the resulting image in 3D Slicer and make sure that the anatomical planes are named correctly by the software. Insert the image into the notebook below.

**Answer:**
![image.png](../assets/notebook-results/mri-cell-17-attachment.png) 

## Part B: Multi-Coil Reconstruction

The variable containing data to be used in this task is named `kspace_coils`.

### B1 (1 pts)

Reconstruct images from the $k$-space data for each coil separately (use your function `k2im` from part A). Compare the coil sensitivities in the variable `coil_sensitivities` with the corresponding reconstructed image. It is sufficient to only compare one slice for each coil-image pair, visualizing e.g. using

```python
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Coil 1
axes[0, 0].imshow(coil_sensitivities[:, :, slice_idx, 0], cmap='gray')
axes[0, 0].set_title('Coil 1 Sensitivity')

# Coil 1 reconstructed image
axes[0, 1].imshow(np.abs(im_coils[:, :, slice_idx, 0]), cmap='gray')
axes[0, 1].set_title('Coil 1 Reconstructed Image')

```

In [ ]:
# YOUR CODE HERE
# Reconstruct images from multi-coil k-space data
k_coils = np.squeeze(kspace_coils)
im_coils = k2im(k_coils)

print(f"Multi-coil k-space shape: {k_coils.shape}")
print(f"Multi-coil reconstructed image shape: {im_coils.shape}")
print(f"Coil sensitivities shape: {coil_sensitivities.shape}")

# Visualize coil sensitivities and reconstructed images
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# YOUR CODE HERE

slice_idx = 150

axes[0, 0].imshow(coil_sensitivities[:, :, slice_idx, 0], cmap='gray')
axes[0, 0].set_title('Coil 1 Sensitivity')
axes[0,0].axis('off')

axes[0, 1].imshow(np.abs(im_coils[:, :, slice_idx, 0]), cmap='gray')
axes[0, 1].set_title('Coil 1 Reconstructed Image')
axes[0,1].axis('off')

axes[1, 0].imshow(np.abs(coil_sensitivities[:, :, slice_idx, 1]), cmap='gray')
axes[1, 0].set_title('Coil 2 Sensitivity')
axes[1, 0].axis('off')

axes[1, 1].imshow(np.abs(im_coils[:, :, slice_idx, 1]), cmap='gray')
axes[1, 1].set_title('Coil 2 Reconstructed Image')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

# Save magnitude images
for i in range(im_coils.shape[3]):
    save_nifti_image(im_coils[:, :, :, i], f'coil_{i+1}_recon', nifti_info)

Multi-coil k-space shape: (192, 129, 192, 2)
Multi-coil reconstructed image shape: (192, 129, 192, 2)
Coil sensitivities shape: (192, 129, 192, 2)


Saved coil_1_recon.nii
Saved coil_2_recon.nii


![Saved output from cell 19, figure 2](../assets/notebook-results/mri-cell-19-figure-1.png)


**Answer:**

There is a correspondence between the coil sensitivity maps and the reconstructed images for each coil.

Coil 1 shows high intensity in the upper region of the FOV. Because of this the reconstructed image decipts the upper part of the brain, but the lower parts fades into darkness as the intensity of the coil lowers. This is also true for coil 2.

### B2 (1 pts)

Combine the different coil images into a single image using a "sum-of-squares reconstruction":

1) Square all pixel values in the images,
2) Sum the images with squared pixel values,
3) Compute the square root of the sum of images,

using a function sum_of_squares_recon.
Save the reconstructed image as NIFTI file and compare it to the reconstructed image in part A. Discuss the differences between the reconstructed images in task A1.

In [10]:
def sum_of_squares_recon(im):
    """
    Compute sum-of-squares reconstruction from multiple coil images.
    
    Steps:
    1. Square all pixel values in the images
    2. Sum the images with squared pixel values
    3. Compute the square root of the sum
    
    Args:
    ----
    im : ndarray
        Complex-valued image from multiple coils (4D array, height x width x depth x n_coils)
    
    Returns:
    -------
    im_sos : ndarray
        The reconstructed image (3D array, height x width x depth)
    """
    # YOUR CODE HERE

    im_sos = np.sqrt(np.sum(np.abs(im)**2, axis=3))
    
    return im_sos


In [11]:
# Compute sum-of-squares reconstruction
im_sos = sum_of_squares_recon(im_coils)

print(f"Sum-of-squares image shape: {im_sos.shape}")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# YOUR CODE HERE

slice_idx = 150

axes[0].imshow(np.abs(im_full[:, :, slice_idx]), cmap='gray')
axes[0].set_title('Original Full Reconstruction (Part A)')
axes[0].axis('off')

axes[1].imshow(np.abs(im_sos[:, :, slice_idx]), cmap='gray')
axes[1].set_title('Sum-of-Squares Reconstruction (Part B)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Save the result
save_nifti_image(im_sos, 'sos_recon', nifti_info)

Sum-of-squares image shape: (192, 129, 192)


Saved sos_recon.nii


![Saved output from cell 23, figure 2](../assets/notebook-results/mri-cell-23-figure-1.png)


**Answer:**

Looking at the Sum-of-Squares (SoS) reconstruction and the original full reconstruction, we can observe a clear visual similarity. The two images are visually nearly identical. The anatomical structures, contrast, and overall intensity distribution in the SoS reconstruction closely match the reference image from Task A1. Although difficult to see with the naked eye in these plots, the Sum-of-Squares combination also seems to improve the Signal-to-Noise-Ratoio (SNR) inside the object compared to individual coils. However, it also changes the statistical distribution of the noise in the background though visually the background levels appear comparable to the reference. The biggest difference is probably the loss of intensity in the middle edges of the SoS image.




### B3 (1 pts): Weighted Sum-of-Squares Reconstruction

If all coils have a low sensitivity at some location in the object (typically in the center) the image obtained from a "sum-of-squares reconstruction" may be hampered by strong intensity variations. Implement a modified version of the coil-combination method called `sum_of_squares_recon_fixed` that uses sensitivity weighting to achieve uniform intensity.

In [12]:
def sum_of_squares_recon_fixed(im, coil_sensitivities):
    """
    Compute sum-of-squares reconstruction with sensitivity weighting.
    
    Args:
    ----
    im : ndarray
        Complex-valued image from multiple coils (4D array, height x width x depth x n_coils)
    ... : ...
        Any other variables you wish to use.
    
    Returns:
    -------
    im_sos_fix : ndarray
        The weighted reconstructed image (3D array, height x width x depth)
    ... : ...
        Any other variables you wish to return.
    """
    # YOUR CODE HERE

    epsilon = 1e-10
    w = np.sqrt(np.sum(np.abs(coil_sensitivities)**2, axis=3))
    
    im_sos = np.sqrt(np.sum(np.abs(im)**2, axis=3))
    
    im_sos_fix = np.divide(im_sos, w + epsilon)
    
    return im_sos_fix, w

In [13]:
# YOUR CODE HERE
# Compute weighted sum-of-squares reconstruction
im_sos_fix, w = sum_of_squares_recon_fixed(im_coils, coil_sensitivities)

print(f"Weighted SOS image shape: {im_sos_fix.shape}")
print(f"Weighting map shape: {w.shape}")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# YOUR CODE HERE

slice_idx = 150

axes[0].imshow(np.abs(im_full[:, :, slice_idx]), cmap='gray')
axes[0].set_title('Reference (Full)')
axes[0].axis('off')

axes[1].imshow(np.abs(im_sos[:, :, slice_idx]), cmap='gray')
axes[1].set_title('Standard SoS')
axes[1].axis('off')

axes[2].imshow(w[:, :, slice_idx], cmap='gray')
axes[2].set_title('Sensitivity Weights')
axes[2].axis('off')

axes[3].imshow(np.abs(im_sos_fix[:, :, slice_idx]), cmap='gray')
axes[3].set_title('Weighted SoS')
axes[3].axis('off')

plt.tight_layout()
plt.show()

save_nifti_image(im_sos_fix, 'weighted_sos_recon', nifti_info)

Weighted SOS image shape: (192, 129, 192)
Weighting map shape: (192, 129, 192)


Saved weighted_sos_recon.nii


![Saved output from cell 26, figure 2](../assets/mri/coil-combination.png)


**Answer:**

In the Sensitivity Weights image we get a clear representation of where the quality of the image is lacking as a result of the coils. In the Weighted SoS we se that the previous faults of the Standard SoS have been improved. This is because after we normalized the intensity of the SoS we get an image that better handles the information of the low-intensity parts that comes at the edges of the coils.

## Part C: Parallel-Imaging Reconstruction (SENSE)

In this part, the $k$-space has been subsampled by skipping every second line (SENSE factor = 2). The variable containing data to be used in this task is named `kspace_subsampled`.

### C1 (1 pts):

Reconstruct images from data from each coil separately. Compare them and their coil sensitivities with the images obtained in task B1. How and why do they differ from the ones in part B?

In [14]:
# YOUR CODE HERE
# Reconstruct from subsampled k-space
k_subsampled = np.squeeze(kspace_subsampled)
im_subsampled = k2im(k_subsampled)

print(f"Subsampled k-space shape: {k_subsampled.shape}")
print(f"Subsampled image shape: {im_subsampled.shape}")

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

slice_idx = 150

axes[0, 0].imshow(np.abs(im_subsampled[:, :, slice_idx, 0]), cmap='gray')
axes[0, 0].set_title("Subsampled Coil 1 (Aliased)")
axes[0, 0].axis('off')

axes[0, 1].imshow(np.abs(im_subsampled[:, :, slice_idx, 1]), cmap='gray')
axes[0, 1].set_title("Subsampled Coil 2 (Aliased)")
axes[0, 1].axis('off')

axes[1, 0].imshow(np.abs(coil_sensitivities[:, :, slice_idx, 0]), cmap='gray')
axes[1, 0].set_title("Coil 1 Sensitivity (Full FOV)")
axes[1, 0].axis('off')

axes[1, 1].imshow(np.abs(coil_sensitivities[:, :, slice_idx, 1]), cmap='gray')
axes[1, 1].set_title("Coil 2 Sensitivity (Full FOV)")
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

#save
plt.imsave("coil1_subsampled.png", np.abs(im_subsampled[:, :, slice_idx, 0]), cmap="gray")
plt.imsave("coil2_subsampled.png", np.abs(im_subsampled[:, :, slice_idx, 1]), cmap="gray")

Subsampled k-space shape: (96, 129, 192, 2)
Subsampled image shape: (96, 129, 192, 2)


![Saved output from cell 28, figure 2](../assets/notebook-results/mri-cell-28-figure-1.png)


**Answer:**
The coil images reconstructed from the subsampled k-space in C1 look very different from the coil images in part B because 
the data has been undersampled. In part B, each coil uses the full k-space, which produces images with the correct 
field-of-view and no aliasing. In C1, however, every second k-space line has been removed (SENSE factor 2). 
When we reconstruct the image with this missing data, two different rows from the full image fold onto the same 
position in the reduced image. This creates the clear aliasing effect we see in the subsampled coil images, 
where anatomical structures appear duplicated or shifted.

The coil sensitivity maps do not show this folding. They are still defined over the full field-of-view and only 
describe how sensitive each coil is at each location, not the anatomical image. Since they are not reconstructed 
from the undersampled data, they remain smooth and unaffected by aliasing.

Overall, the main difference is that the coil images in C1 are folded because of missing k-space lines, while the 
images in part B are fully sampled. This difference is exactly why a SENSE reconstruction is needed in the next step.


### C2 (2 pts):

Implement a SENSE reconstruction to unfold aliased images using coil sensitivity information and solving pixel-wise linear systems. Remember that the images obtained from task C1 are folded, while the coil sensitivity maps are given in the full field of view without folding.

**Hint 1:** The function `np.mod()` is useful when dealing with folding.

**Hint 2:** You can solve a linear system $AX = B$ in NumPy using `np.linalg.lstsq(A, B)`.

In [ ]:
def SENSE(im_folded, coil_sensitivities):
    """
    Reconstruct an MR image using the SENSE algorithm.
    
    SENSE factor = 2 case: unfold aliased images using coil sensitivities.
    
    Args:
    ----
    im_folded : ndarray
        A folded image from parallel imaging (4D array, height x width x depth x n_coils)
    coil_sensitivities : ndarray
        The sensitivities for each coil (4D array, height x width x depth x n_coils)
    
    Returns:
    -------
    im : ndarray
        A reconstructed image (3D array, height x width x depth)
    """
    # YOUR CODE HERE
    rows_folded, cols, slices, n_coils = im_folded.shape
    rows_full = rows_folded * 2
    
    im_recon = np.zeros((rows_full, cols, slices), dtype=complex)
    
    shift = rows_full // 4

    for z in range(slices):
        sens_slice = coil_sensitivities[:, :, z, :]
        data_slice = im_folded[:, :, z, :]
        
        for x in range(cols):
            for y in range(rows_folded):
                pos1 = (y + shift) % rows_full
                pos2 = (y + 3 * shift) % rows_full
                
                S = sens_slice[[pos1, pos2], x, :].T
                m_vec = data_slice[y, x, :]
                
                rho = np.linalg.lstsq(S, m_vec, rcond=None)[0]
                
                im_recon[pos1, x, z] = rho[0]
                im_recon[pos2, x, z] = rho[1]
                
    return im_recon

# Perform SENSE reconstruction
print("Running SENSE reconstruction (this may take a moment)...")
im_sense = SENSE(im_subsampled, coil_sensitivities)

print(f"SENSE reconstructed image shape: {im_sense.shape}")

# Visualize comparison
slice_idx = 150 
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(np.abs(im_subsampled[:, :, slice_idx, 0]), cmap='gray')
axes[0].set_title("Folded Image (Coil 1)")
axes[0].axis('off')

axes[1].imshow(np.abs(im_subsampled[:, :, slice_idx, 1]), cmap='gray')
axes[1].set_title("Folded Image (Coil 2)")
axes[1].axis('off')

axes[2].imshow(np.abs(im_sense[:, :, slice_idx]), cmap='gray')
axes[2].set_title("SENSE Reconstructed (Unfolded)")
axes[2].axis('off')

plt.tight_layout()
plt.show()

# Save the result
save_nifti_image(im_sense, 'sense_recon', nifti_info)

Running SENSE reconstruction (this may take a moment)...
SENSE reconstructed image shape: (192, 129, 192)


Saved sense_recon.nii


![Saved output from cell 31, figure 2](../assets/mri/sense-reconstruction.png)


**Answer:**


The SENSE reconstruction in C2 unfolds the aliased images from C1 and recovers a full field of view. Compared to the folded coil images, the SENSE result places anatomical structures in the correct spatial locations, showing that the sensitivity information is successfully used to separate the overlapping rows. The reconstruction is not as clean as the fully sampled image from part A, but the main brain features match well, which indicates that the unfolding works despite the reduced data.

The remaining artifacts come from the fact that the subsampled k-space contains less information and that the provided sensitivity maps are smooth and synthetic. Because of this, SENSE cannot perfectly correct all distortions. Still, the C2 result is a clear improvement over the folded images and demonstrates how parallel imaging can recover a usable full-FOV image from undersampled data.